# smoke_test.ipynb — Colab End-to-End Pipeline Tests (CT1-CT8)

Run all cells in order. Each cell ends with `print('CTn PASSED')`.  
If any assertion fails, fix the issue before proceeding to training.

| Test | Description |
|------|-------------|
| CT1  | Colab env setup: Drive mount, git clone, pip install, imports, create Drive dirs |
| CT2  | Process single song (row 0) end-to-end via `process_song()` |
| CT3  | `batch_ingest.py` on 2 enabled rows → consolidated manifest |
| CT4  | Re-run `batch_ingest.py` → idempotency (both songs skipped, no duplicate rows) |
| CT5  | `split_dataset.py` → train/val/test splits, no song-level leakage |
| CT6  | Demucs separation path: `separate_stems=True` → 4 stem WAVs + correct mel/PR shapes |
| CT7  | Augmentation: raw WAV segment → 3 variants (pitch-only, time-only, combined) → vocode → validate f0 ratio |
| CT8  | Postprocessing: BigVGAN vocode + `compute_f1` + `compute_fad` + mel/PR visualization |


---

### §37 cell-header convention

Every code cell in this notebook is preceded by a markdown header that
follows the project standard (ENGINEERING_DECISIONS §37, see
[colab/README.md](README.md)):

> **What this does.** plain-English summary.  
> **Inputs.** files / variables / env read.  
> **Outputs.** files / variables / state written.  
> **Action required.** anything the user must edit, click, approve, or run
> elsewhere. Prefix the heading with **⚠** when the action requires switching
> machines (e.g. F1 backfill needs `basic_pitch_env` locally).  
> **Runtime.** order-of-magnitude (seconds / minutes / hours).

Stub headers below carry `TODO` placeholders — fill them in when you next
edit the cell. `colab/postprocessing.ipynb` is the reference implementation.


<!-- §37-stub -->
## Cell 1 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# CT1 — Colab environment setup
# ============================================================
# BEFORE running this cell:
#   1. 🔑 Secrets tab → add secret  GITHUB_TOKEN  (Classic PAT, repo scope)
#
#   2. YouTube cookies (needed for CT2/CT3 downloads):
#      a. On your local machine, install the Chrome extension
#         "Get cookies.txt LOCALLY"  (search Chrome Web Store)
#      b. Open youtube.com while logged in to your Google account
#      c. Click the extension icon → "Export" → save as  youtube_cookies.txt
#      d. Upload the file to Google Drive at:
#         MyDrive/MusicProject/youtube_cookies.txt
# ============================================================

# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repo using GitHub token (skip if already present)
import os
from google.colab import userdata

REPO_DIR = '/content/MusicProject'
GH_TOKEN = userdata.get('GITHUB_TOKEN')

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GH_TOKEN}@github.com/galgeva1/MusicProject.git'
    !git clone -q {clone_url} {REPO_DIR}
    print('Repo cloned.')
else:
    print('Repo already cloned — pulling latest...')
    !cd {REPO_DIR} && git pull -q

assert os.path.exists(REPO_DIR), 'Clone failed — check GITHUB_TOKEN secret'
%cd {REPO_DIR}

# 3. Install Node.js (required by yt-dlp to solve YouTube JS signature challenges)
print('Installing Node.js for yt-dlp signature solver...')
!apt-get install -y -q nodejs 2>/dev/null | tail -1

# 4. Install dependencies
!pip install -q -U yt-dlp          # always latest — YouTube changes frequently
!pip install -q -r requirements_colab.txt
!pip install -q --no-deps "git+https://github.com/spotify/basic-pitch.git@main"
!pip install -q resampy mir_eval

# Verify basic-pitch installed correctly
import importlib
assert importlib.util.find_spec('basic_pitch') is not None, \
    'basic_pitch not installed — check the pip output above'
print('basic_pitch OK')

# Verify Node.js available for yt-dlp
import shutil
node_path = shutil.which('node')
assert node_path, 'node not found on PATH — apt-get install nodejs failed'
print(f'Node.js: {node_path}')

# 5. Path constants
DRIVE_ROOT   = '/content/drive/MyDrive/MusicProject'
DATA_ROOT    = f'{DRIVE_ROOT}/MusicProjectData'
MANIFEST_DIR = f'{DRIVE_ROOT}/data'

# 6. Create canonical Drive dirs (idempotent)
import pathlib
for d in [
    DATA_ROOT,
    MANIFEST_DIR,
    f'{DRIVE_ROOT}/checkpoints',
    f'{DRIVE_ROOT}/logs',
    f'{DRIVE_ROOT}/slakh_raw',
    f'{DRIVE_ROOT}/slakh_processed',
]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
print('Drive dirs created/verified.')

# Copy batch_songs.csv to Drive if not already there
import shutil
batch_csv_drive = f'{DRIVE_ROOT}/batch_songs.csv'
if not os.path.exists(batch_csv_drive):
    shutil.copy(f'{REPO_DIR}/batch_songs.csv', batch_csv_drive)
    print('Copied batch_songs.csv to Drive.')

# 7. Set YouTube cookies env var (bypasses bot detection on Colab)
COOKIES_FILE = f'{DRIVE_ROOT}/youtube_cookies.txt'
if os.path.exists(COOKIES_FILE):
    os.environ['YTDLP_COOKIES_FILE'] = COOKIES_FILE
    print(f'YouTube cookies loaded: {COOKIES_FILE}')
else:
    print('WARNING: youtube_cookies.txt not found on Drive.')
    print('  YouTube downloads will fail with bot-detection error.')
    print(f'  Upload your cookies file to: {COOKIES_FILE}')
    print('  See cell comment above for instructions.')

# 8. Verify imports
import sys
sys.path.insert(0, REPO_DIR)
from process_song_offline import process_song

# Assertions
for d in [DATA_ROOT, MANIFEST_DIR, f'{DRIVE_ROOT}/checkpoints', f'{DRIVE_ROOT}/logs']:
    assert os.path.isdir(d), f'Missing Drive dir: {d}'
assert callable(process_song), 'process_song not callable'
assert os.environ.get('YTDLP_COOKIES_FILE'), \
    'YTDLP_COOKIES_FILE not set — upload youtube_cookies.txt to Drive first'

print('CT1 PASSED')


<!-- §37-stub -->
## Cell 2 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# CT2 — Validate pre-processed data uploaded from local machine
#
# BEFORE running this cell, the local pipeline must have been run:
#   python preprocessing/batch_ingest.py \
#       --csv batch_songs.csv \
#       --out_root MusicProjectData \
#       --manifest_out MusicProjectData/dataset_manifest.csv \
#       --log MusicProjectData/batch_ingest_log.csv \
#       --upload_to_drive 1idAndcTUg95IXqM8r0FtvehM6xl1z7Cl
# ============================================================

import csv, pathlib, torch

manifest_path = pathlib.Path(DATA_ROOT) / 'dataset_manifest.csv'
assert manifest_path.exists(), (
    f'dataset_manifest.csv not found at {manifest_path}\n'
    f'Run batch_ingest.py locally with --upload_to_drive first.'
)

with open(manifest_path, newline='', encoding='utf-8') as f:
    manifest_rows = list(csv.DictReader(f))

assert manifest_rows, 'dataset_manifest.csv is empty'
song_ids = {r['song_id'] for r in manifest_rows}
print(f'Manifest: {len(manifest_rows)} segments across {len(song_ids)} songs')
print(f'Songs: {sorted(song_ids)}')

# Validate first segment of each song
checked = set()
for row in manifest_rows:
    sid = row['song_id']
    if sid in checked:
        continue
    checked.add(sid)

    mel_path = pathlib.Path(DATA_ROOT) / row['segment_path'].replace('\\', '/')
    pr_path  = pathlib.Path(DATA_ROOT) / row['score_path'].replace('\\', '/')

    assert mel_path.exists(), f'Missing mel: {mel_path}'
    assert pr_path.exists(),  f'Missing piano_roll: {pr_path}'

    mel = torch.load(mel_path, weights_only=True)
    pr  = torch.load(pr_path,  weights_only=True)

    assert mel.shape == (80, 430),     f'{sid}: bad mel shape {mel.shape}'
    assert pr.shape  == (2, 128, 430), f'{sid}: bad PR shape {pr.shape}'
    assert mel.min().item() >= -1.01,  f'{sid}: mel min out of range'
    assert mel.max().item() <=  1.01,  f'{sid}: mel max out of range'

    print(f'{sid}: mel={mel.shape}  PR={pr.shape}  range=[{mel.min():.3f}, {mel.max():.3f}]  ✓')

print('CT2 PASSED')


<!-- §37-stub -->
## Cell 3 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# CT3 — Validate manifest completeness and log
# (batch_ingest was run locally; we just verify the uploaded outputs)
# ============================================================

import csv, pathlib

manifest_path = pathlib.Path(DATA_ROOT) / 'dataset_manifest.csv'
log_path      = pathlib.Path(DATA_ROOT) / 'batch_ingest_log.csv'

# Manifest checks
assert manifest_path.exists(), f'dataset_manifest.csv missing: {manifest_path}'
with open(manifest_path, newline='', encoding='utf-8') as f:
    manifest_rows = list(csv.DictReader(f))

song_ids = {r['song_id'] for r in manifest_rows}
assert len(song_ids) >= 1, f'Expected >= 1 song, got {song_ids}'

seg_paths = [r['segment_path'] for r in manifest_rows]
assert len(seg_paths) == len(set(seg_paths)), 'Duplicate segment_path in manifest!'

print(f'Manifest: {len(manifest_rows)} segments, {len(song_ids)} songs — no duplicates ✓')

# Log checks
assert log_path.exists(), f'batch_ingest_log.csv missing: {log_path}'
with open(log_path, newline='', encoding='utf-8') as f:
    log_rows = list(csv.DictReader(f))

failed = [r for r in log_rows if r.get('status') == 'failed']
assert not failed, f'Failed rows in log: {failed}'
statuses = {r['status'] for r in log_rows}
print(f'Log: {len(log_rows)} rows, statuses: {statuses} — no failures ✓')

CT3_MANIFEST_COUNT = len(manifest_rows)
print('CT3 PASSED')


<!-- §37-stub -->
## Cell 4 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# CT4 — Re-run batch_ingest → idempotency (all skipped)
# ============================================================

import subprocess, csv, pathlib

batch_csv_drive = f'{DRIVE_ROOT}/batch_songs.csv'
manifest_out   = f'{MANIFEST_DIR}/dataset_manifest.csv'
resolved_csv   = f'{DRIVE_ROOT}/batch_songs_resolved.csv'
log_csv_ct4    = f'{DRIVE_ROOT}/batch_ingest_log_ct4.csv'

cmd = [
    'python', 'preprocessing/batch_ingest.py',
    '--csv',          batch_csv_drive,
    '--out_root',     DATA_ROOT,
    '--manifest_out', manifest_out,
    '--resolved_csv', resolved_csv,
    '--log',          log_csv_ct4,
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
assert result.returncode == 0, f'batch_ingest.py failed with code {result.returncode}'

# Check all skipped
with open(log_csv_ct4, newline='', encoding='utf-8') as f:
    log_rows = list(csv.DictReader(f))
non_skipped = [r for r in log_rows if r.get('status') != 'skipped']
assert not non_skipped, f'Expected all skipped, got: {[r["status"] for r in non_skipped]}'
print(f'All {len(log_rows)} rows skipped.')

# Check no duplicate segment_path in consolidated manifest
with open(manifest_out, newline='', encoding='utf-8') as f:
    manifest_rows = list(csv.DictReader(f))
seg_paths = [r['segment_path'] for r in manifest_rows]
assert len(seg_paths) == len(set(seg_paths)), 'Duplicate segment_path in manifest!'
print(f'Manifest still has {len(manifest_rows)} rows, no duplicates.')

print('CT4 PASSED')

<!-- §37-stub -->
## Cell 5 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# CT5 — split_dataset on Colab manifest
# ============================================================

import subprocess, csv, pathlib

manifest_out = f'{MANIFEST_DIR}/dataset_manifest.csv'

cmd = [
    'python', 'preprocessing/split_dataset.py',
    '--manifest', manifest_out,
    '--out_dir',  MANIFEST_DIR,
    '--train', '0.8',
    '--val',   '0.1',
    '--test',  '0.1',
    '--seed',  '42',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
assert result.returncode == 0, f'split_dataset.py exited with code {result.returncode}'

# Load splits
def load_csv(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

train_rows = load_csv(f'{MANIFEST_DIR}/train.csv')
val_rows   = load_csv(f'{MANIFEST_DIR}/val.csv')
test_rows  = load_csv(f'{MANIFEST_DIR}/test.csv')
manifest_rows = load_csv(manifest_out)

# Segment count matches
total_split = len(train_rows) + len(val_rows) + len(test_rows)
assert total_split == len(manifest_rows), (
    f'Segment count mismatch: split total {total_split} != manifest {len(manifest_rows)}'
)

# No song-level leakage between train and val/test
train_songs = {r['song_id'] for r in train_rows}
val_songs   = {r['song_id'] for r in val_rows}
test_songs  = {r['song_id'] for r in test_rows}
assert train_songs.isdisjoint(val_songs),  f'Song leakage train/val: {train_songs & val_songs}'
assert train_songs.isdisjoint(test_songs), f'Song leakage train/test: {train_songs & test_songs}'
assert val_songs.isdisjoint(test_songs),   f'Song leakage val/test: {val_songs & test_songs}'

print(f'train={len(train_rows)} | val={len(val_rows)} | test={len(test_rows)} | total={total_split}')
print(f'train songs: {train_songs}')
print(f'val   songs: {val_songs}')
print(f'test  songs: {test_songs}')
print('CT5 PASSED')

<!-- §37-stub -->
## Cell 6 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# CT6 — Source separation path (Demucs htdemucs_ft)
# Expected runtime: ~10-20 min (Demucs, GPU preferred)
# ============================================================

import csv, pathlib, torch

batch_csv_drive = f'{DRIVE_ROOT}/batch_songs.csv'
with open(batch_csv_drive, newline='', encoding='utf-8-sig') as f:
    rows = [r for r in csv.DictReader(f) if r.get('enabled', 'false').lower() == 'true']

row = rows[0]
print(f'Processing (separate_stems=True): {row["artist"]} — {row["song_name"]}')

# Run with Demucs source separation enabled
result = process_song(row, out_root=pathlib.Path(DATA_ROOT), separate_stems=True,
                      skip_if_exists=False)
print(f'Status: {result["status"]}  |  {result.get("error", "")}')
assert result['status'] == 'ok', f'process_song (sep) failed: {result.get("error")}'

song_dir = pathlib.Path(result['song_dir'])

# Assert all 4 stems exist directly under stems/
stems_dir = song_dir / 'stems'
assert stems_dir.exists(), f'stems/ dir missing: {stems_dir}'
expected_stems = {'vocals', 'drums', 'bass', 'other'}
present = {p.stem for p in stems_dir.glob('*.wav')}
missing = expected_stems - present
assert not missing, f'Missing stems: {missing}  (found: {sorted(present)})'
print(f'Stems found: {sorted(present)}')

# Assert mel / PR shapes still correct
mels_dir = song_dir / 'processed_data' / 'mels'
prs_dir  = song_dir / 'processed_data' / 'piano_rolls'
seg_mel = sorted(mels_dir.glob('segment_*.pt'))[0]
seg_pr  = sorted(prs_dir.glob('segment_*.pt'))[0]

mel = torch.load(seg_mel, weights_only=True)
pr  = torch.load(seg_pr,  weights_only=True)
assert mel.shape == (80, 430),     f'Bad mel shape: {mel.shape}'
assert pr.shape  == (2, 128, 430), f'Bad PR shape:  {pr.shape}'
assert mel.min().item() >= -1.01,  f'mel min out of range: {mel.min():.4f}'
assert mel.max().item() <=  1.01,  f'mel max out of range: {mel.max():.4f}'

print(f'mel={mel.shape}  PR={pr.shape}  range=[{mel.min():.3f}, {mel.max():.3f}]')
print('CT6 PASSED')


<!-- §37-stub -->
## Cell 7 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# CT7 — Augmentation: WAV segment → 3 variants → vocode → validate
#
# Verifies our "4× data" logic:
#   For 1 processed segment (mel + piano roll derived from raw WAV),
#   JointAugment can produce 3 distinct valid variants:
#     Variant A: pitch-only  (+1 or +2 semitones)
#     Variant B: time-only   (±10% stretch)
#     Variant C: combined    (pitch + time + spec augment)
# Each variant is vocoded and validated.  Pitch-shifted variants
# are also checked for a correct f0 ratio vs the original.
# ============================================================

import csv, pathlib, random, sys
import numpy as np
import torch
import soundfile as sf
import librosa
from IPython.display import Audio, display

sys.path.insert(0, REPO_DIR)
from preprocessing.augmentation import JointAugment
from postprocessing.bigvgan_vocoder import BigVGANVocoder

# ── Load a segment from the manifest (CT2/CT3 output) ─────────────────────────
manifest_path = f'{MANIFEST_DIR}/dataset_manifest.csv'
with open(manifest_path, newline='', encoding='utf-8') as f:
    manifest_rows = list(csv.DictReader(f))

seg = manifest_rows[0]
mel_orig = torch.load(pathlib.Path(DATA_ROOT) / seg['segment_path'], weights_only=True).float()
pr_orig  = torch.load(pathlib.Path(DATA_ROOT) / seg['score_path'],   weights_only=True).float()
print(f'Source segment: mel={mel_orig.shape}  PR={pr_orig.shape}')
print(f'  mel range=[{mel_orig.min():.3f}, {mel_orig.max():.3f}]')

# ── 3 augmentation configs ─────────────────────────────────────────────────────
AUG_CONFIGS = {
    'pitch_only': {
        'enabled': True,
        'pitch_shift':  {'p': 1.0, 'max_semitones': 2},
        'time_stretch': {'p': 0.0, 'max_pct': 0.10},
        'spec_augment': {'p': 0.0, 'time_mask_max': 30, 'freq_mask_max': 8, 'n_time': 0, 'n_freq': 0},
    },
    'time_only': {
        'enabled': True,
        'pitch_shift':  {'p': 0.0, 'max_semitones': 2},
        'time_stretch': {'p': 1.0, 'max_pct': 0.10},
        'spec_augment': {'p': 0.0, 'time_mask_max': 30, 'freq_mask_max': 8, 'n_time': 0, 'n_freq': 0},
    },
    'combined': {
        'enabled': True,
        'pitch_shift':  {'p': 1.0, 'max_semitones': 2},
        'time_stretch': {'p': 1.0, 'max_pct': 0.10},
        'spec_augment': {'p': 1.0, 'time_mask_max': 30, 'freq_mask_max': 8, 'n_time': 2, 'n_freq': 2},
    },
}

# Apply 3 augmentations with fixed seeds for reproducibility
SEEDS = {'pitch_only': 0, 'time_only': 1, 'combined': 2}
variants = {'original': (mel_orig.clone(), pr_orig.clone())}

print()
for name, cfg in AUG_CONFIGS.items():
    aug = JointAugment(cfg)
    random.seed(SEEDS[name])
    mel_aug, pr_aug = aug(mel_orig.clone(), pr_orig.clone())
    variants[name] = (mel_aug, pr_aug)

    assert mel_aug.shape == (80, 430),     f'{name}: bad mel shape {mel_aug.shape}'
    assert pr_aug.shape  == (2, 128, 430), f'{name}: bad PR shape {pr_aug.shape}'
    assert mel_aug.min().item() >= -1.01,  f'{name}: mel min out of range'
    assert mel_aug.max().item() <=  1.01,  f'{name}: mel max out of range'
    assert not torch.isnan(mel_aug).any(), f'{name}: mel has NaN'
    print(f'{name}: mel={mel_aug.shape}  range=[{mel_aug.min():.3f}, {mel_aug.max():.3f}]')

print('\nAll shape / range / NaN checks passed.')

# ── Vocode all 4 variants via BigVGAN 22kHz ───────────────────────────────────
device  = 'cuda' if torch.cuda.is_available() else 'cpu'
vocoder = BigVGANVocoder(model_name='bigvgan_22k', device=device)
SR = 22050

out_dir = pathlib.Path(f'{DRIVE_ROOT}/ct7_aug_wavs')
out_dir.mkdir(parents=True, exist_ok=True)

wavs = {}
print()
for name, (m, _) in variants.items():
    audio    = vocoder.mel_to_audio(m)
    wav_path = out_dir / f'{name}.wav'
    sf.write(str(wav_path), audio, SR)
    wavs[name] = audio
    print(f'Vocoded {name}: {audio.shape[0]} samples → {wav_path.name}')

# ── f0 ratio check for pitch-shifted variants ─────────────────────────────────
def _median_f0(audio, sr):
    f0, _, _ = librosa.pyin(audio.astype(np.float64), fmin=65.0, fmax=2100.0, sr=sr)
    valid = f0[~np.isnan(f0)]
    return float(np.median(valid)) if len(valid) > 0 else float('nan')

f0_orig = _median_f0(wavs['original'], SR)
print(f'\nf0 original: {f0_orig:.2f} Hz')

for name in ['pitch_only', 'combined']:
    f0_aug = _median_f0(wavs[name], SR)
    if not np.isnan(f0_orig) and not np.isnan(f0_aug) and f0_orig > 0:
        ratio = f0_aug / f0_orig
        print(f'f0 {name}: {f0_aug:.2f} Hz  ratio={ratio:.4f}')
        # Any ±2-semitone shift lands in [0.89, 1.13]; use ±6% tolerance
        assert 0.83 < ratio < 1.20, \
            f'{name} f0 ratio {ratio:.4f} outside expected range [0.83, 1.20]'
        print(f'  → f0 ratio OK ✓')
    else:
        print(f'f0 {name}: f0_orig={f0_orig:.2f}  f0_aug={f0_aug:.2f} (pitch unclear — skipping ratio assert)')

# ── Listen-check: display audio widgets ──────────────────────────────────────
print()
for name, audio in wavs.items():
    print(f'─── {name} ───')
    display(Audio(audio, rate=SR))

print('\nCT7 PASSED')


<!-- §37-stub -->
## Cell 8 — TODO short title

**What this does.** TODO

**Inputs.** TODO

**Outputs.** TODO

**Action required.** TODO (or _none_)

**Runtime.** TODO (seconds / minutes / hours)


In [ ]:
# ============================================================
# CT8 — Postprocessing: F1, FAD, mel/PR visualization
# ============================================================

import csv, pathlib, shutil, sys
import numpy as np
import torch
import soundfile as sf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.insert(0, REPO_DIR)
from postprocessing.bigvgan_vocoder import BigVGANVocoder
from postprocessing.f1_eval import compute_f1
from postprocessing.fad_eval import compute_fad

# ── Load segment 0 from CT2/CT3 manifest ──────────────────────────────────────
manifest_path = f'{MANIFEST_DIR}/dataset_manifest.csv'
with open(manifest_path, newline='', encoding='utf-8') as f:
    manifest_rows = list(csv.DictReader(f))

seg      = manifest_rows[0]
mel      = torch.load(pathlib.Path(DATA_ROOT) / seg['segment_path'], weights_only=True).float()
pr       = torch.load(pathlib.Path(DATA_ROOT) / seg['score_path'],   weights_only=True).float()
song_dir = pathlib.Path(DATA_ROOT) / pathlib.Path(seg['segment_path']).parts[0]
song_name = seg['song_name']
midi_path = song_dir / f'{song_name}.mid'
assert midi_path.exists(), f'Reference MIDI not found: {midi_path}'
print(f'mel={mel.shape}  PR={pr.shape}  MIDI={midi_path.name}')

# ── Vocode segment → WAV ──────────────────────────────────────────────────────
device  = 'cuda' if torch.cuda.is_available() else 'cpu'
vocoder = BigVGANVocoder(model_name='bigvgan_22k', device=device)
SR = 22050

ct8_dir = pathlib.Path(f'{DRIVE_ROOT}/ct8_postproc')
ct8_dir.mkdir(parents=True, exist_ok=True)

audio   = vocoder.mel_to_audio(mel)
gen_wav = ct8_dir / 'segment_gen.wav'
sf.write(str(gen_wav), audio, SR)
print(f'Saved: {gen_wav.name}  ({audio.shape[0]} samples @ {SR} Hz)')

# ── F1 Evaluation ─────────────────────────────────────────────────────────────
print('\n--- F1 Evaluation ---')
f1_result = compute_f1(generated_wav=gen_wav, reference_midi=midi_path)
print(f'  F1:        {f1_result["f1"]:.4f}')
print(f'  Precision: {f1_result["precision"]:.4f}  |  Recall: {f1_result["recall"]:.4f}')
print(f'  Notes: predicted={f1_result["n_predicted"]}  reference={f1_result["n_reference"]}  matched={f1_result["matched"]}')
assert 'f1' in f1_result,                   'compute_f1 result missing "f1" key'
assert 0.0 <= f1_result['f1'] <= 1.0,       f'f1 out of [0, 1]: {f1_result["f1"]}'

# ── FAD Evaluation ────────────────────────────────────────────────────────────
# Real dir: original vocoded WAV
# Generated dir: CT7 augmented WAVs (pitch_only / time_only / combined)
ct7_dir = pathlib.Path(f'{DRIVE_ROOT}/ct7_aug_wavs')
fad_real_dir = ct8_dir / 'fad_real'
fad_real_dir.mkdir(exist_ok=True)
shutil.copy(str(gen_wav), str(fad_real_dir / 'real_seg.wav'))

if ct7_dir.exists() and list(ct7_dir.glob('*.wav')):
    print('\n--- FAD Evaluation ---')
    fad_score = compute_fad(str(fad_real_dir), str(ct7_dir))
    assert np.isfinite(fad_score), f'FAD returned non-finite value: {fad_score}'
    print(f'  FAD score: {fad_score:.4f}  (finite ✓)')
else:
    print('\n--- FAD Evaluation --- SKIPPED (CT7 wavs not found)')

# ── Mel + Piano Roll Visualization ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

im0 = axes[0].imshow(mel.numpy(), aspect='auto', origin='lower', cmap='magma')
axes[0].set_title(f'Mel spectrogram  |  {song_name}  seg-0')
axes[0].set_xlabel('Time frames')
axes[0].set_ylabel('Mel bins')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(pr[0].numpy(), aspect='auto', origin='lower', cmap='Blues')
axes[1].set_title(f'Piano roll (onset)  |  {song_name}  seg-0')
axes[1].set_xlabel('Time frames')
axes[1].set_ylabel('MIDI pitch')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
viz_path = ct8_dir / 'mel_pr_viz.png'
plt.savefig(str(viz_path), dpi=100)
plt.show()
assert viz_path.exists() and viz_path.stat().st_size > 0, 'Visualization PNG not saved'
print(f'Saved: {viz_path.name}')

print('\nCT8 PASSED')
